# Validate Reconstructions: Ankle Challenge

Use this notebook to validate ankle challenge reconstructions against the reference ankle challenge folder.

Checks performed:
- file names and folder structure must match 100%
- candidate file keys must be exactly `ismrmrd_header` and `reconstruction_rss`
- kspace is explicitly checked and reported (`kspace` must not be present)
- `reconstruction_rss` shape must match the reference for every file
- candidate total size must be <= 500 MB

This notebook only validates and reports. It does not write output files.


In [ ]:
from pathlib import Path
import shutil
from IPython.display import Markdown, display
from validate_reconstructions_core import run_validation_report, export_reconstruction_only_folder


In [ ]:
# Configure\nREFERENCE_ROOT = Path("MosaicMRI/anatomy_generalization_challenge/ankle")\nMY_ROOT = Path("MosaicMRI/anatomy_generalization_challenge/ankle_reconstructions")\nMAX_FOLDER_SIZE_GIB = 500 / 1024  # 500 MB\n

ZIP_OUTPUT_PATH = MY_ROOT.parent / f"{MY_ROOT.name}.zip"
print("REFERENCE_ROOT:", REFERENCE_ROOT)\nprint("MY_ROOT:", MY_ROOT)\nprint("MAX_FOLDER_SIZE_GIB:", MAX_FOLDER_SIZE_GIB)\n
print("ZIP_OUTPUT_PATH:", ZIP_OUTPUT_PATH)


In [ ]:
READY_TO_PACKAGE, REPORT = run_validation_report(
    reference_root=REFERENCE_ROOT,
    my_root=MY_ROOT,
    max_folder_size_gib=MAX_FOLDER_SIZE_GIB,
)


In [ ]:
# Step 2: if kspace is present, optionally export a clean folder (reconstruction_rss + ismrmrd_header only)
kspace_count = int(REPORT["summary"].get("kspace_present_count", 0))
if kspace_count == 0:
    display(Markdown("### No k-space found"))
    print("Candidate folder already has no kspace datasets.")
else:
    display(Markdown("### K-space detected: optional clean export"))
    print(f"Detected {kspace_count} files containing kspace.")
    CLEAN_ROOT = MY_ROOT.parent / f"{MY_ROOT.name}_reconstructions_only"
    print("Suggested clean folder:", CLEAN_ROOT)
    ans = input("Create clean folder without kspace now? Type yes to continue: ").strip().lower()
    if ans == "yes":
        result = export_reconstruction_only_folder(
            source_root=MY_ROOT,
            output_root=CLEAN_ROOT,
            overwrite=False,
        )
        print("Export complete:", result)
        ans2 = input("Use clean folder as MY_ROOT and re-run validation now? Type yes to continue: ").strip().lower()
        if ans2 == "yes":
            MY_ROOT = CLEAN_ROOT
            ZIP_OUTPUT_PATH = MY_ROOT.parent / f"{MY_ROOT.name}.zip"
            READY_TO_PACKAGE, REPORT = run_validation_report(
                reference_root=REFERENCE_ROOT,
                my_root=MY_ROOT,
                max_folder_size_gib=MAX_FOLDER_SIZE_GIB,
            )
    else:
        print("Skipped clean export.")


In [ ]:
# Step 3: offer ZIP only if folder is ready (names/shapes/keys OK, no kspace, and size within limit)
if "REPORT" not in globals() or "READY_TO_PACKAGE" not in globals():
    display(Markdown("### Run validation first"))
    print("Run the validation cell first.")
else:
    s = REPORT["summary"]
    kspace_ok = int(s.get("kspace_present_count", 0)) == 0
    size_ok = float(s.get("size_gib", 1e9)) <= float(MAX_FOLDER_SIZE_GIB)
    can_zip = bool(READY_TO_PACKAGE) and kspace_ok and size_ok

    if not can_zip:
        display(Markdown("### ZIP blocked"))
        print("ZIP is not available yet.")
        if not kspace_ok:
            print("- kspace is still present. Use Step 2 clean export first.")
        if not size_ok:
            print(f"- size is above limit: {s["size_gib"]:.3f} GiB > {MAX_FOLDER_SIZE_GIB:.3f} GiB")
        if not bool(s.get("name_match_100", False)):
            print("- file names/structure are not a 100% match.")
    else:
        display(Markdown("### Ready to ZIP"))
        print("Folder is ready. ZIP target:", ZIP_OUTPUT_PATH)
        ans = input("Create zip now? Type yes to continue: ").strip().lower()
        if ans == "yes":
            ZIP_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
            if ZIP_OUTPUT_PATH.exists():
                ZIP_OUTPUT_PATH.unlink()
            shutil.make_archive(
                str(ZIP_OUTPUT_PATH.with_suffix("")),
                "zip",
                root_dir=str(MY_ROOT.parent),
                base_dir=str(MY_ROOT.name),
            )
            print("ZIP created:", ZIP_OUTPUT_PATH)
        else:
            print("Canceled.")
